In [1]:
import pandas as pd
import numpy as np
import re, string, os, sys, time

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
)
import joblib

# Make backend.utils importable from notebooks/
sys.path.insert(0, os.path.abspath(".."))
from backend.utils import preprocess_text, map_labels, heuristics_score

In [2]:
# Load dataset
df = pd.read_csv("../data/fakereviews.csv")
print("Shape:", df.shape)
df.head()

Shape: (40432, 4)


,category,rating,label,text_
0,Home_and_Kitchen_5,5.0,CG,"Love this! Well made, sturdy, and very comfor..."
1,Home_and_Kitchen_5,5.0,CG,"love it, a great upgrade from the original. I..."
2,Home_and_Kitchen_5,5.0,CG,This pillow saved my back. I love the look and...
3,Home_and_Kitchen_5,1.0,CG,"Missing information on how to use it, but it i..."
4,Home_and_Kitchen_5,5.0,CG,Very nice set. Good quality. We have had the s...


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40432 entries, 0 to 40431
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   category  40432 non-null  object 
 1   rating    40432 non-null  float64
 2   label     40432 non-null  object 
 3   text_     40432 non-null  object 
dtypes: float64(1), object(3)
memory usage: 1.2+ MB


In [4]:
# Map labels: CG → fake, OR → real
df['label'] = map_labels(df['label'])
df['label'].value_counts()

label
fake    20216
real    20216
Name: count, dtype: int64

In [5]:
# Keep only the columns we need
df = df[['text_', 'label']]
df.head()

,text_,label
0,"Love this! Well made, sturdy, and very comfor...",fake
1,"love it, a great upgrade from the original. I...",fake
2,This pillow saved my back. I love the look and...,fake
3,"Missing information on how to use it, but it i...",fake
4,Very nice set. Good quality. We have had the s...,fake


In [6]:
# Preprocess text using the shared utility function
# (lowercases, removes URLs, punctuation, digits-only tokens, stopwords)
df['clean_text'] = df['text_'].apply(preprocess_text)
print("Sample cleaned text:")
print(df['clean_text'].iloc[0][:200])

Sample cleaned text:
love well made sturdy comfortable love itvery pretty


In [7]:
# Train / Test split (stratified, reproducible)
X = df['clean_text']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42,
)
print(f"Train: {len(X_train)}  |  Test: {len(X_test)}")

Train: 32345  |  Test: 8087


In [8]:
# Build a pipeline: TfidfVectorizer → LogisticRegression
# Uses same hyperparams as backend/train.py for consistency
pipeline = make_pipeline(
    TfidfVectorizer(max_features=5000, ngram_range=(1, 2), min_df=3, max_df=0.9),
    LogisticRegression(class_weight='balanced', max_iter=2000, random_state=42),
)

t0 = time.time()
pipeline.fit(X_train, y_train)
elapsed = time.time() - t0
print(f"Training completed in {elapsed:.2f}s")

Training completed in 6.41s


In [9]:
# Evaluate on test set
y_pred = pipeline.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print()
print(classification_report(y_test, y_pred, digits=4))

Accuracy: 0.8894522072461976

              precision    recall  f1-score   support

        fake     0.8941    0.8835    0.8888      4044
        real     0.8849    0.8954    0.8901      4043

    accuracy                         0.8895      8087
   macro avg     0.8895    0.8895    0.8894      8087
weighted avg     0.8895    0.8895    0.8894      8087



In [10]:
# Confusion matrix & ROC AUC
cm = confusion_matrix(y_test, y_pred, labels=['fake', 'real'])
print("Confusion Matrix [fake, real]:")
print(cm)

# ROC AUC (need probabilities)
fake_idx = list(pipeline.classes_).index('fake')
probs_fake = pipeline.predict_proba(X_test)[:, fake_idx]
y_true_bin = (y_test == 'fake').astype(int)
auc = roc_auc_score(y_true_bin, probs_fake)
print(f"\nROC AUC: {auc:.4f}")

Confusion Matrix [fake, real]:
[[3573  471]
 [ 423 3620]]

ROC AUC: 0.9588


In [11]:
# Top 20 features for the 'fake' class
vectorizer = pipeline.named_steps['tfidfvectorizer']
clf = pipeline.named_steps['logisticregression']
feature_names = np.array(vectorizer.get_feature_names_out())
coeffs = clf.coef_[fake_idx]
top_indices = np.argsort(coeffs)[-20:][::-1]

print("Top 20 features for 'fake' class:")
for i, idx in enumerate(top_indices, 1):
    print(f"  {i:>2}. {feature_names[idx]:<30s}  coeff={coeffs[idx]:.4f}")

Top 20 features for 'fake' class:
   1. even                            coeff=5.5633
   2. though                          coeff=4.4325
   3. without                         coeff=4.3814
   4. much                            coeff=3.7085
   5. however                         coeff=3.7064
   6. another                         coeff=3.6424
   7. every                           coeff=3.5377
   8. could                           coeff=3.5111
   9. wont                            coeff=3.3364
  10. others                          coeff=3.2715
  11. actually                        coeff=3.2295
  12. end                             coeff=3.2282
  13. really                          coeff=3.1784
  14. far                             coeff=3.1483
  15. easily                          coeff=3.1225
  16. plus                            coeff=3.1147
  17. otherwise                       coeff=3.0696
  18. instead                         coeff=2.9843
  19. quite                           coeff=2.96

In [12]:
# Quick demo: predict on sample reviews
FAKE_THRESHOLD = 0.85
REAL_THRESHOLD = 0.35

samples = [
    "This product is amazing and works perfectly! Best purchase ever!!!",
    "Decent product, does the job. Battery drains after about 3 hours.",
    "I received this product for free in exchange for my honest review.",
]

for text in samples:
    cleaned = preprocess_text(text)
    prob = pipeline.predict_proba([cleaned])[0][fake_idx]
    h = heuristics_score(text)
    combined = 0.75 * prob + 0.25 * h['heuristic_score']
    if combined >= FAKE_THRESHOLD:
        label = "Fake"
    elif combined <= REAL_THRESHOLD:
        label = "Real"
    else:
        label = "Uncertain"
    print(f"[{label}]  fake_prob={prob:.4f}  combined={combined:.4f}  |  {text[:80]}...")

[Real]  fake_prob=0.2923  combined=0.3308  |  This product is amazing and works perfectly! Best purchase ever!!!...
[Real]  fake_prob=0.1936  combined=0.1452  |  Decent product, does the job. Battery drains after about 3 hours....
[Uncertain]  fake_prob=0.8026  combined=0.6966  |  I received this product for free in exchange for my honest review....


In [13]:
# Save the pipeline to models/
os.makedirs("../models", exist_ok=True)
pipeline_path = "../models/fake_review_pipeline.pkl"
joblib.dump(pipeline, pipeline_path)
print(f"Pipeline saved to {pipeline_path}")

Pipeline saved to ../models/fake_review_pipeline.pkl
